# Qwen3-30B-A3B-Instruct-2507 gốc, 2-shot — hai khuôn ngữ cảnh

Notebook **mỏng có chủ ý**: mọi logic nằm trong `finetune/kaggle_30b_eval.py` (trong
git, có xuất xứ commit). Notebook chỉ điều phối ba ô. Vá logic bằng cell trên Kaggle
chính là chuyện đã làm số liệu phiên 1 FT-03 không khớp bất kỳ commit nào
(`gate_base_model.md` §6) — đừng lặp lại.

**Cấu hình panel phải — BẮT BUỘC:** Accelerator `GPU T4 x2` · Internet `On` ·
Persistence `Files only` · Visibility `Private`

`GPU T4 x2` không phải tuỳ chọn cho nhanh: GGUF Q4_K_M của 30B-A3B nặng **18.6 GB**,
cộng KV cache của `n_ctx=16384` thì vượt xa 16 GB của một T4. Thiếu card thứ hai,
llama.cpp đẩy phần thừa xuống RAM CPU — vẫn chạy nhưng chậm hàng chục lần, 137 câu
sẽ không kịp trong 12 giờ của Kaggle. Chặng `prep` có **cổng chặn VRAM** nạp thử
model rồi đọc `nvidia-smi`; không thấy trọng số trên **cả hai** card thì nó trả mã
lỗi và dừng, thay vì để bạn đốt vài giờ GPU rồi mới biết.

**Kaggle Secret bắt buộc** (đọc qua `kaggle_secrets`, KHÔNG hardcode): chỉ `HF_TOKEN`.

---

| Ô | Mô hình sinh | n_shot | Ngữ cảnh |
|---|---|---|---|
| 1 | Qwen3-30B-A3B-Instruct-2507 (Q4_K_M) | 2 | GraphRAG — `results_graphrag_final1_20260729-022916.json` |
| 2 | Qwen3-30B-A3B-Instruct-2507 (Q4_K_M) | 2 | Naive RAG — `results_baseline_20260710-085236.json` |

GraphRAG chạy trước vì đó là ô có giá trị khoa học cao hơn — session đứt thì phần
quan trọng đã xong.

**Đúng hai file ngữ cảnh của lượt 4B và 8B**, cùng bộ tham số sinh
(`gate_base_model.md` §3), cùng `n_shot = 2` → cột Δ giữa ba lượt phản ánh **đúng
một biến là cỡ mô hình**.

**Ngân sách giờ.** Chặng `run` mặc định dừng sau **10 giờ** (Kaggle cắt ở 12h, chừa
2h để chạy `table` và tải kết quả về). Mỗi câu in kèm timestamp, thời gian đã trôi,
tốc độ giây/câu và ETA — nhìn vài câu đầu là biết 137 câu có kịp hay không.

In [ ]:
import os,subprocess;from kaggle_secrets import UserSecretsClient as S;s=S();os.environ.update(HF_TOKEN=s.get_secret("HF_TOKEN"),HF_XET_HIGH_PERFORMANCE="1");R="/kaggle/working/repo";B="dev/fine-tune";U="https://github.com/tandat-dao/vn-legal-graphrag.git";r=subprocess.run((f"git -C {R} fetch -q --all && git -C {R} checkout -q {B} && git -C {R} pull -q --ff-only" if os.path.isdir(R+"/.git") else f"git clone -q -b {B} {U} {R}")+f" && git -C {R} log -1 --format='commit %H%n  %s'",shell=True,capture_output=True,text=True);print(r.stdout+r.stderr,">>> GHI COMMIT HASH NÀY VÀO KHÓA LUẬN")

In [ ]:
!cd /kaggle/working/repo && python finetune/kaggle_30b_eval.py --stage prep

In [ ]:
!cd /kaggle/working/repo && python finetune/kaggle_30b_eval.py --stage run ; python finetune/kaggle_30b_eval.py --stage table

**Vì sao ô trên dùng `;` chứ không phải `&&`.** Chặng `run` trả mã 2 khi chạm hạn
giờ. Với `&&` thì `table` sẽ không chạy và bạn mất luôn bảng số của phần đã làm
được. Dấu `;` cho `table` chạy trong mọi trường hợp.

---

**Session đứt hoặc chạm hạn giờ?** Chạy lại ô 1 → ô 2 → ô 3 trong session mới.
`replay.py` ghi từng câu vào `.partial.jsonl` rồi `flush()` ngay, và chặng `run`
luôn truyền `--resume`, nên lần chạy sau tiếp đúng chỗ dở — chỉ mất câu đang sinh
giữa chừng. Muốn chạy đúng một ô: `--stage run --cells 2`.

**Đừng bao giờ gọi `replay.py` tay mà quên `--resume`** — nó sẽ `unlink()` file
`.partial.jsonl`, tức xoá sạch tiến độ đã chạy.

**Kết quả dang dở vẫn đọc được.** `replay.py` chỉ viết file JSON đầu ra khi chạy
hết 137 câu; đứt giữa chừng thì chỉ còn `.partial.jsonl`. Chặng `table` của script
này đọc được **cả hai dạng**, và đánh dấu rõ hàng nào là dang dở kèm cảnh báo không
so trực tiếp với hàng 4B/8B (khác mẫu số).

**Muốn nới/siết ngân sách giờ:** `--stage run --gio-toi-da 8`.

**Lần chạy đầu chưa ghim sha256 GGUF.** Repo `unsloth/Qwen3-30B-A3B-Instruct-2507-GGUF`
chưa từng dùng ở dự án này. `prep` tính sha256 rồi ghi vào
`finetune/reports/30b_artifacts.json`; **lần sau truyền `--gguf-sha256 <giá trị đó>`**
để biến thành cổng chặn thật.

**Tải kết quả về trước khi session hết hạn:**
`finetune/results/results_{graphrag,baseline}_30b-base-s2.json` (và `.partial.jsonl`
nếu dang dở) — script này **không** tự đẩy lên HF.